<a href="https://colab.research.google.com/github/akbar260/resumeMatcher/blob/main/extractionAndParsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pdfplumber transformers accelerate sentencepiece

In [6]:

import pdfplumber
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json

# Load the model once
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)


def extract_and_parse(pdf_path):
    # -------------------------
    # Step 1: Extract text
    # -------------------------
    text = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"

    # -------------------------
    # Step 2: Prompt Qwen
    # -------------------------
    prompt = f"""
You are an expert resume parser.

Extract the following information from the resume.

Return ONLY valid JSON.

Fields:
- name
- email
- phone
- skills
- education
- work_experience
- certifications
- projects

Resume:

{text}
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    chat = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(chat, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )
    parsed_output = extract_and_parse("/content/Akbar resume.pdf")

    print(json.dumps(parsed_output, indent=4))

    # -------------------------
    # Step 3: Parse JSON
    # -------------------------
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        return {
            "error": "Model did not return valid JSON.",
            "raw_output": response
        }

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [5]:
import pdfplumber
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json

# -------------------------------
# Load Qwen model (only once)
# -------------------------------
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!\n")


def extract_and_parse(pdf_path):

    # -------------------------------
    # Step 1: Extract text from PDF
    # -------------------------------
    print("Extracting text from PDF...")

    text = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"



    # -------------------------------
    # Step 2: Prompt the model
    # -------------------------------
    prompt = f"""
Parse the following resume into a structred JSON.

Resume:

{text}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat,
        return_tensors="pt"
    ).to(model.device)

    # -------------------------------
    # Step 3: Generate response
    # -------------------------------
    print("Generating response from Qwen...\n")

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            temperature=0.5
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    print("Generation complete.\n")

    # -------------------------------
    # Step 4: Convert to JSON
    # -------------------------------
    try:
        parsed = json.loads(response)
        return parsed

    except Exception:

        print("Model did not return valid JSON.\n")

        return {
            "raw_output": response
        }


# ------------------------------------
# Call the function
# ------------------------------------

parsed_output = extract_and_parse("/HuzaifaNaveed (1).pdf")

print("\nFinal Output:\n")

print(json.dumps(parsed_output, indent=4))

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!

Extracting text from PDF...
Generating response from Qwen...

Generation complete.

Model did not return valid JSON.


Final Output:

{
    "raw_output": "```json\n{\n  \"name\": \"Huzaifa Naveed\",\n  \"location\": {\n    \"city\": \"Karachi\",\n    \"country\": \"Pakistan\"\n  },\n  \"email\": \"huzaifanaveed2020@gmail.com\",\n  \"phone\": \"+92-318-2340485\",\n  \"social_links\": [\n    {\n      \"platform\": \"LinkedIn\",\n      \"url\": \"https://www.linkedin.com/in/huzaifa-naveedd/\"\n    },\n    {\n      \"platform\": \"GitHub\",\n      \"url\": \"https://github.com/HuzaifaaNaveed\"\n    },\n    {\n      \"platform\": \"Hugging Face\",\n      \"url\": \"https://huggingface.co/huzaifanaveeddd\"\n    }\n  ],\n  \"education\": {\n    \"degree\": \"Bachelor of Science in Artificial Intelligence\",\n    \"institute\": \"FAST NUCES Karachi\",\n    \"city\": \"Karachi, Pakistan\",\n    \"start_date\": \"Aug 2022\",\n    \"end_date\": \"June 2026\",\n    \"gp

In [3]:
def refine_output(parsed_json):

    prompt = f"""
You are an expert data validation and resume normalization system.

You will receive JSON extracted from a resume.

Your task is NOT to summarize the resume.
Your task is ONLY to clean, validate and normalize the JSON.

Rules:

1. Return ONLY valid JSON.
2. Do not add markdown.
3. Do not add explanations.
4. Do not invent any information.
5. Remove duplicate entries.
6. Remove unnecessary text.
7. Remove repeated work experiences.
8. Remove repeated education entries.
9. Remove repeated skills.
10. Remove empty strings.
11. Replace missing values with null.
12. If a project appears inside work experience, move it to projects.
13. If certifications are actually skills, move them to skills.
14. Keep only concise descriptions.
15. Normalize phone numbers if possible.
16. Normalize email addresses.
17. Keep company names exactly as written.
18. Keep degree names exactly as written.
19. Remove any hallucinated fields.
20. Do NOT change correct information.

Return the cleaned JSON.

Input JSON:

{json.dumps(parsed_json, indent=2)}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            temperature=0.0
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    try:
        return json.loads(response)

    except Exception:
        return {
            "raw_output": response
        }

In [4]:
parsed_output = extract_and_parse("/HuzaifaNaveed (1).pdf")

refined_output = refine_output(parsed_output)

print(json.dumps(refined_output, indent=4))

Extracting text from PDF...
Generating response from Qwen...

Generation complete.

Model did not return valid JSON.

{
    "raw_output": "```json\n{\n  \"name\": \"Huzaifa Naveed\",\n  \"email\": \"huzaifanaveed2020@gmail.com\",\n  \"phone\": \"+92-318-2340485\",\n  \"skills\": [\n    \"Machine Learning\",\n    \"Artificial Neural Networks\",\n    \"Natural Language Processing\",\n    \"Computer Vision\",\n    \"Generative AI\",\n    \"Recommender Systems\",\n    \"DevOps\",\n    \"Data Structures & Algorithms\"\n  ],\n  \"education\": [\n    {\n      \"institution\": \"FAST NUCES Karachi, Pakistan\",\n      \"degree\": \"Bachelor of Science in Artificial Intelligence\",\n      \"cgpa\": \"3.90/4.00\",\n      \"start_date\": \"Aug 2022\",\n      \"end_date\": \"June 2026\"\n    }\n  ],\n  \"work_experience\": [\n    {\n      \"company_name\": \"Unikrew Solutions Karachi, Pakistan\",\n      \"position\": \"Deep Learning Engineer\",\n      \"start_date\": \"June 2026\",\n      \"end_dat

In [6]:
pip install docling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 656.0/656.0 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.1/282.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.1 MB/s eta 0:00:00
   ━━━━

In [7]:
from docling.document_converter import DocumentConverter

source = "/content/HuzaifaNaveed (1).pdf"
converter = DocumentConverter()
doc = converter.convert(source).document
print(doc.export_to_markdown())

[INFO] 2026-07-20 11:56:37,861 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-20 11:56:37,874 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-07-20 11:56:37,893 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.1/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-20 11:56:38,733 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-07-20 11:56:39,296 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-20 11:56:39,303 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-20 11:56:40,405 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-20 11:56:40,409 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-07-20 11:56:40,412 [RapidOCR] download_file.py:68: Init

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

## Huzaifa Naveed

Karachi, Pakistan | huzaifanaveed2020@gmail.com | +92-318-2340485

linkedin.com/in/huzaifa-naveedd | github.com/HuzaifaaNaveed | huggingface.co/huzaifanaveeddd

## Education

## FAST NUCES Karachi

Bachelor of Science in Artificial Intelligence, CGPA: 3.90/4.00

Karachi, Pakistan

Aug 2022 - June 2026

- Relevant Coursework: Machine Learning, Artificial Neural Networks, , Natural Language Processing, Computer Vision, Generative AI, Recommender Systems, DevOps, Data Structures &amp; Algorithms

## Experience

## Unikrew Solutions

## Deep Learning Engineer

Karachi, Pakistan

June 2026 - Present

- Built a 3D face reconstruction pipeline using 3D Gaussian Splatting with FLAME mesh initialization via MICA and PnP-based pose estimation for photorealistic novel view synthesis.
- Implemented a thumbprints and signatures detection pipeline using YOLO26 combined with handcrafted rules.
- Fine-tuned SmolVLM on a custom dataset to create a robust bilingual English-Urdu OCR ex

In [8]:
from docling.document_converter import DocumentConverter

source = "/content/HuzaifaNaveed (1).pdf"
converter = DocumentConverter()
doc = converter.convert(source).document

# Markdown (what you already have)
markdown = doc.export_to_markdown()

# Plain text
text = doc.export_to_text()

# HTML
html = doc.export_to_html()

# Lossless JSON / dict (full schema: text, layout, provenance, bounding boxes, etc.)
data_dict = doc.export_to_dict()

# DocTags — structure + spatial tags, useful for downstream ML/VLM pipelines
doctags = doc.export_to_document_tokens()

[INFO] 2026-07-20 11:58:42,684 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-20 11:58:42,685 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-07-20 11:58:42,784 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-20 11:58:42,785 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-20 11:58:44,296 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-20 11:58:44,298 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-07-20 11:58:44,309 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-07-20 11:58:44,310 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

/tmp/ipykernel_25973/1729509707.py:20: DeprecationWarning: Use export_to_doctags() instead.
  doctags = doc.export_to_document_tokens()


In [11]:
doctags

"<doctag><section_header_level_1><loc_205><loc_23><loc_295><loc_31>Huzaifa Naveed</section_header_level_1>\n<text><loc_126><loc_35><loc_374><loc_41>Karachi, Pakistan | huzaifanaveed2020@gmail.com | +92-318-2340485</text>\n<text><loc_77><loc_44><loc_423><loc_50>linkedin.com/in/huzaifa-naveedd | github.com/HuzaifaaNaveed | huggingface.co/huzaifanaveeddd</text>\n<section_header_level_1><loc_29><loc_61><loc_81><loc_65>Education</section_header_level_1>\n<section_header_level_1><loc_38><loc_71><loc_142><loc_77>FAST NUCES Karachi</section_header_level_1>\n<text><loc_38><loc_80><loc_264><loc_86>Bachelor of Science in Artificial Intelligence, CGPA: 3.90/4.00</text>\n<text><loc_388><loc_71><loc_458><loc_77>Karachi, Pakistan</text>\n<text><loc_377><loc_80><loc_457><loc_86>Aug 2022 - June 2026</text>\n<unordered_list><list_item><loc_50><loc_88><loc_469><loc_101>Relevant Coursework: Machine Learning, Artificial Neural Networks, , Natural Language Processing, Computer Vision, Generative AI, Recomme